In [1]:
print("Let's Start")

Let's Start


In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error , r2_score

from sklearn.decomposition import PCA

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression , Ridge , Lasso
from sklearn.tree import DecisionTreeRegressor , ExtraTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor , RandomForestRegressor , AdaBoostRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import mlflow
import mlflow.sklearn
import dagshub

In [4]:
df = pd.read_csv('/mnt/WindowsD/Projects/real-estate-valuation/data/processed/feature_selection/gurgaon_properties_post_feature_selection_v2.csv')
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 7,0.45,2,2,1,Relatively New,1000.0,0,0,2,Low,Mid Floor
1,flat,sector 3,0.50,2,2,1,Old Property,722.0,0,0,0,Low,Low Floor
2,flat,sector 61,1.47,2,2,2,Under Construction,1333.0,0,0,2,Medium,Low Floor
3,flat,sector 92,0.70,2,2,3,Under Construction,1218.0,0,0,2,Low,Mid Floor
4,flat,sector 36,0.41,2,2,3,New Property,654.0,0,0,2,Low,Mid Floor


In [5]:
df.isna().sum()

property_type      0
sector             0
price              0
bedRoom            0
bathroom           0
balcony            0
agePossession      0
built_up_area      0
servant room       0
store room         0
furnishing_type    0
luxury_category    0
floor_category     0
dtype: int64

In [16]:
x = df.drop(columns=['price'])
y = df['price']
y_transformed = np.log1p(y)

In [17]:
from sklearn.model_selection import train_test_split
x_train , x_test , y_train , y_test= train_test_split(x , y, test_size=0.2 , random_state=42)

In [24]:
try:
    dagshub.init(repo_owner='jubito27', repo_name='gurgaon-estate', mlflow=True)
    mlflow.set_tracking_uri('https://dagshub.com/jubito27/gurgaon-estate.mlflow')
except Exception as e:
    print("Error occured as :" , e)
mlflow.set_experiment('xgboost-pipeline')

Accessing as jubito27

Initialized MLflow to track repo "jubito27/gurgaon-estate"

Repository jubito27/gurgaon-estate initialized!

2026/11/21 11:42:27 INFO mlflow.tracking.fluent: Experiment with name 'xgboost-pipeline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/ba3a864514044442bd1b07f25dda514e', creation_time=1795241548465, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1795241548465, lifecycle_stage='active', name='xgboost-pipeline', tags={}, trace_location=None, workspace='default'>

In [25]:
import json
import tempfile
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)

from xgboost import XGBRegressor
from category_encoders import TargetEncoder


# -----------------------------
# MLflow configuration
# -----------------------------
REGISTERED_MODEL_NAME = "xgboost-pipeline"




# -----------------------------
# Data preparation
# -----------------------------

X = df.drop(columns=["price"])
y = df["price"]

# Train on log-transformed target because predictions are later
# converted back using np.expm1().
y_transformed = np.log1p(y)


xtrain, xtest, ytrain_transformed, ytest_transformed = train_test_split(
    X,
    y_transformed,
    test_size=0.2,
    random_state=42,
)

# Keep original-price test target for business-friendly metrics.
ytest_original = np.expm1(ytest_transformed)


# -----------------------------
# Feature configuration
# -----------------------------

numeric_columns = [
    "bedRoom",
    "bathroom",
    "built_up_area",
    "servant room",
    "store room",
]

ordinal_columns = [
    "property_type",
    "balcony",
    "furnishing_type",
    "luxury_category",
    "floor_category",
]

onehot_columns = ["agePossession"]
target_encoding_columns = ["sector"]

model_params = {'n_estimators': 974,
'learning_rate': 0.10526914056016663,
'max_depth': 12,
'min_child_weight': 1,
'gamma': 0.018795185277108897,
'subsample': 0.8079122956973779,
'colsample_bytree': 0.7947938787822193,
'reg_alpha': 4.17187378088139e-05,
'reg_lambda': 2.1089841197246224e-07}


# -----------------------------
# Preprocessor
# -----------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_columns,
        ),
        (
            "ordinal",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
            ordinal_columns,
        ),
        (
            "age_onehot",
            OneHotEncoder(
                drop="first",
                sparse_output=False,
                handle_unknown="ignore",
            ),
            onehot_columns,
        ),
        (
            "sector_target",
            TargetEncoder(),
            target_encoding_columns,
        ),
    ],
    remainder="passthrough",
)


# -----------------------------
# Full pipeline
# -----------------------------

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", XGBRegressor(**model_params)),
    ]
)


# -----------------------------
# MLflow run
# -----------------------------

with mlflow.start_run(
    run_name="xgboost_log_target_pipeline",
    tags={
        "project": "real_estate_valuation",
        "model_type": "XGBoost",
        "target_transformation": "log1p",
        "pipeline": "preprocessor+xgboost",
    },
) as run:

    run_id = run.info.run_id

    # -------------------------
    # Log parameters
    # -------------------------

    mlflow.log_params(
        {
            **{f"xgboost_{key}": value for key, value in model_params.items()},
            "test_size": 0.2,
            "random_state": 42,
            "target_transformation": "log1p",
            "prediction_transformation": "expm1",
            "n_training_rows": xtrain.shape[0],
            "n_test_rows": xtest.shape[0],
            "n_features": X.shape[1],
        }
    )

    # Log feature configuration as tags for easy filtering in MLflow UI.
    mlflow.set_tags(
        {
            "numeric_columns": ", ".join(numeric_columns),
            "ordinal_columns": ", ".join(ordinal_columns),
            "onehot_columns": ", ".join(onehot_columns),
            "target_encoding_columns": ", ".join(target_encoding_columns),
        }
    )

    # -------------------------
    # Train pipeline
    # -------------------------

    pipeline.fit(xtrain, ytrain_transformed)

    # -------------------------
    # Predict and inverse transform
    # -------------------------

    ypred_log = pipeline.predict(xtest)
    ypred_original = np.expm1(ypred_log)

    # Prevent negative predicted prices after inverse transformation.
    ypred_original = np.maximum(ypred_original, 0)

    # -------------------------
    # Calculate scores
    # -------------------------

    mae_log = mean_absolute_error(ytest_transformed, ypred_log)
    mse_log = mean_squared_error(ytest_transformed, ypred_log)
    rmse_log = np.sqrt(mse_log)
    r2_log = r2_score(ytest_transformed, ypred_log)

    mae_original = mean_absolute_error(ytest_original, ypred_original)
    mse_original = mean_squared_error(ytest_original, ypred_original)
    rmse_original = np.sqrt(mse_original)
    r2_original = r2_score(ytest_original, ypred_original)
    mape_original = mean_absolute_percentage_error(
        ytest_original,
        ypred_original,
    )

    # -------------------------
    # Log metrics
    # -------------------------

    mlflow.log_metrics(
        {
            "mae_log_target": mae_log,
            "mse_log_target": mse_log,
            "rmse_log_target": rmse_log,
            "r2_log_target": r2_log,
            "mae_original_price": mae_original,
            "mse_original_price": mse_original,
            "rmse_original_price": rmse_original,
            "r2_original_price": r2_original,
            "mape_original_price": mape_original,
        }
    )

    print(
        f"""
        Log-target metrics:
        MAE: {mae_log:.4f}
        RMSE: {rmse_log:.4f}
        R2: {r2_log:.4f}

        Original-price metrics:
        MAE: {mae_original:,.2f}
        RMSE: {rmse_original:,.2f}
        R2: {r2_original:.4f}
        MAPE: {mape_original:.2%}
        """
    )

    # -------------------------
    # Create model signature
    # -------------------------

    # Use a small sample from training features.
    input_example = xtrain.head(5)

    # Pipeline predicts log(price), so this is the model output.
    output_example = pipeline.predict(input_example)

    signature = mlflow.models.infer_signature(
        model_input=input_example,
        model_output=output_example,
    )

    # -------------------------
    # Save run configuration artifact
    # -------------------------

    run_config = {
        "numeric_columns": numeric_columns,
        "ordinal_columns": ordinal_columns,
        "onehot_columns": onehot_columns,
        "target_encoding_columns": target_encoding_columns,
        "model_params": model_params,
        "target": "price",
        "target_transformation": "log1p",
        "test_size": 0.2,
        "random_state": 42,
        "metrics": {
            "mae_log_target": mae_log,
            "rmse_log_target": rmse_log,
            "r2_log_target": r2_log,
            "mae_original_price": mae_original,
            "rmse_original_price": rmse_original,
            "r2_original_price": r2_original,
            "mape_original_price": mape_original,
        },
    }

    with tempfile.TemporaryDirectory() as temp_dir:
        config_path = Path(temp_dir) / "run_config.json"

        with open(config_path, "w", encoding="utf-8") as file:
            json.dump(run_config, file, indent=4, default=str)

        mlflow.log_artifact(str(config_path), artifact_path="run_metadata")

    # -------------------------
    # Save test predictions artifact
    # -------------------------

    predictions_df = xtest.copy()
    predictions_df["actual_price"] = ytest_original.to_numpy()
    predictions_df["predicted_price"] = ypred_original
    predictions_df["actual_log_price"] = ytest_transformed.to_numpy()
    predictions_df["predicted_log_price"] = ypred_log

    with tempfile.TemporaryDirectory() as temp_dir:
        predictions_path = Path(temp_dir) / "test_predictions.csv"

        predictions_df.to_csv(predictions_path, index=False)

        mlflow.log_artifact(
            str(predictions_path),
            artifact_path="predictions",
        )

    # -------------------------
    # Log the complete pipeline
    # -------------------------

    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="real_estate_price_pipeline",
        signature=signature,
        input_example=input_example,
        registered_model_name=REGISTERED_MODEL_NAME,
        serialization_format="cloudpickle",
    )

    # Log the model URI for easy reference.
    mlflow.log_param("logged_model_uri", model_info.model_uri)

    print(f"MLflow run ID: {run_id}")
    print(f"Logged model URI: {model_info.model_uri}")


        Log-target metrics:
        MAE: 0.1136
        RMSE: 0.1725
        R2: 0.9037

        Original-price metrics:
        MAE: 0.45
        RMSE: 0.91
        R2: 0.8826
        MAPE: 21.33%
        


/mnt/WindowsD/Projects/real-estate-valuation/.estate/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/21 11:42:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, 

MLflow run ID: fc01fd5dcd1f4fcc8c9907d83cf8cdac
Logged model URI: models:/m-2097489b265b4f6eb6a07c65959f5078
🏃 View run xgboost_log_target_pipeline at: https://dagshub.com/jubito27/gurgaon-estate.mlflow/#/experiments/1/runs/fc01fd5dcd1f4fcc8c9907d83cf8cdac
🧪 View experiment at: https://dagshub.com/jubito27/gurgaon-estate.mlflow/#/experiments/1


In [26]:
try:
    dagshub.init(repo_owner='jubito27', repo_name='gurgaon-estate', mlflow=True)
    mlflow.set_tracking_uri('https://dagshub.com/jubito27/gurgaon-estate.mlflow')
except Exception as e:
    print("Error occured as :" , e)
mlflow.set_experiment('randomforest-pipeline')

Initialized MLflow to track repo "jubito27/gurgaon-estate"

Repository jubito27/gurgaon-estate initialized!

2026/11/21 11:47:38 INFO mlflow.tracking.fluent: Experiment with name 'randomforest-pipeline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/add267245ddb475dbbf1888e8bdb5718', creation_time=1795241859406, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1795241859406, lifecycle_stage='active', name='randomforest-pipeline', tags={}, trace_location=None, workspace='default'>

In [27]:
import json
import tempfile
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)

from sklearn.ensemble import RandomForestRegressor
from category_encoders import TargetEncoder


# -----------------------------
# MLflow configuration
# -----------------------------
REGISTERED_MODEL_NAME = "randomforest-pipeline"




# -----------------------------
# Data preparation
# -----------------------------

X = df.drop(columns=["price"])
y = df["price"]

# Train on log-transformed target because predictions are later
# converted back using np.expm1().
y_transformed = np.log1p(y)


xtrain, xtest, ytrain_transformed, ytest_transformed = train_test_split(
    X,
    y_transformed,
    test_size=0.2,
    random_state=42,
)

# Keep original-price test target for business-friendly metrics.
ytest_original = np.expm1(ytest_transformed)


# -----------------------------
# Feature configuration
# -----------------------------

numeric_columns = [
    "bedRoom",
    "bathroom",
    "built_up_area",
    "servant room",
    "store room",
]

ordinal_columns = [
    "property_type",
    "balcony",
    "furnishing_type",
    "luxury_category",
    "floor_category",
]

onehot_columns = ["agePossession"]
target_encoding_columns = ["sector"]

model_params = {'n_estimators': 876,
 'max_depth': 19,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 1.0,
 'bootstrap': True}


# -----------------------------
# Preprocessor
# -----------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_columns,
        ),
        (
            "ordinal",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
            ordinal_columns,
        ),
        (
            "age_onehot",
            OneHotEncoder(
                drop="first",
                sparse_output=False,
                handle_unknown="ignore",
            ),
            onehot_columns,
        ),
        (
            "sector_target",
            TargetEncoder(),
            target_encoding_columns,
        ),
    ],
    remainder="passthrough",
)


# -----------------------------
# Full pipeline
# -----------------------------

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(**model_params)),
    ]
)


# -----------------------------
# MLflow run
# -----------------------------

with mlflow.start_run(
    run_name="xgboost_log_target_pipeline",
    tags={
        "project": "real_estate_valuation",
        "model_type": "XGBoost",
        "target_transformation": "log1p",
        "pipeline": "preprocessor+xgboost",
    },
) as run:

    run_id = run.info.run_id

    # -------------------------
    # Log parameters
    # -------------------------

    mlflow.log_params(
        {
            **{f"xgboost_{key}": value for key, value in model_params.items()},
            "test_size": 0.2,
            "random_state": 42,
            "target_transformation": "log1p",
            "prediction_transformation": "expm1",
            "n_training_rows": xtrain.shape[0],
            "n_test_rows": xtest.shape[0],
            "n_features": X.shape[1],
        }
    )

    # Log feature configuration as tags for easy filtering in MLflow UI.
    mlflow.set_tags(
        {
            "numeric_columns": ", ".join(numeric_columns),
            "ordinal_columns": ", ".join(ordinal_columns),
            "onehot_columns": ", ".join(onehot_columns),
            "target_encoding_columns": ", ".join(target_encoding_columns),
        }
    )

    # -------------------------
    # Train pipeline
    # -------------------------

    pipeline.fit(xtrain, ytrain_transformed)

    # -------------------------
    # Predict and inverse transform
    # -------------------------

    ypred_log = pipeline.predict(xtest)
    ypred_original = np.expm1(ypred_log)

    # Prevent negative predicted prices after inverse transformation.
    ypred_original = np.maximum(ypred_original, 0)

    # -------------------------
    # Calculate scores
    # -------------------------

    mae_log = mean_absolute_error(ytest_transformed, ypred_log)
    mse_log = mean_squared_error(ytest_transformed, ypred_log)
    rmse_log = np.sqrt(mse_log)
    r2_log = r2_score(ytest_transformed, ypred_log)

    mae_original = mean_absolute_error(ytest_original, ypred_original)
    mse_original = mean_squared_error(ytest_original, ypred_original)
    rmse_original = np.sqrt(mse_original)
    r2_original = r2_score(ytest_original, ypred_original)
    mape_original = mean_absolute_percentage_error(
        ytest_original,
        ypred_original,
    )

    # -------------------------
    # Log metrics
    # -------------------------

    mlflow.log_metrics(
        {
            "mae_log_target": mae_log,
            "mse_log_target": mse_log,
            "rmse_log_target": rmse_log,
            "r2_log_target": r2_log,
            "mae_original_price": mae_original,
            "mse_original_price": mse_original,
            "rmse_original_price": rmse_original,
            "r2_original_price": r2_original,
            "mape_original_price": mape_original,
        }
    )

    print(
        f"""
        Log-target metrics:
        MAE: {mae_log:.4f}
        RMSE: {rmse_log:.4f}
        R2: {r2_log:.4f}

        Original-price metrics:
        MAE: {mae_original:,.2f}
        RMSE: {rmse_original:,.2f}
        R2: {r2_original:.4f}
        MAPE: {mape_original:.2%}
        """
    )

    # -------------------------
    # Create model signature
    # -------------------------

    # Use a small sample from training features.
    input_example = xtrain.head(5)

    # Pipeline predicts log(price), so this is the model output.
    output_example = pipeline.predict(input_example)

    signature = mlflow.models.infer_signature(
        model_input=input_example,
        model_output=output_example,
    )

    # -------------------------
    # Save run configuration artifact
    # -------------------------

    run_config = {
        "numeric_columns": numeric_columns,
        "ordinal_columns": ordinal_columns,
        "onehot_columns": onehot_columns,
        "target_encoding_columns": target_encoding_columns,
        "model_params": model_params,
        "target": "price",
        "target_transformation": "log1p",
        "test_size": 0.2,
        "random_state": 42,
        "metrics": {
            "mae_log_target": mae_log,
            "rmse_log_target": rmse_log,
            "r2_log_target": r2_log,
            "mae_original_price": mae_original,
            "rmse_original_price": rmse_original,
            "r2_original_price": r2_original,
            "mape_original_price": mape_original,
        },
    }

    with tempfile.TemporaryDirectory() as temp_dir:
        config_path = Path(temp_dir) / "run_config.json"

        with open(config_path, "w", encoding="utf-8") as file:
            json.dump(run_config, file, indent=4, default=str)

        mlflow.log_artifact(str(config_path), artifact_path="run_metadata")

    # -------------------------
    # Save test predictions artifact
    # -------------------------

    predictions_df = xtest.copy()
    predictions_df["actual_price"] = ytest_original.to_numpy()
    predictions_df["predicted_price"] = ypred_original
    predictions_df["actual_log_price"] = ytest_transformed.to_numpy()
    predictions_df["predicted_log_price"] = ypred_log

    with tempfile.TemporaryDirectory() as temp_dir:
        predictions_path = Path(temp_dir) / "test_predictions.csv"

        predictions_df.to_csv(predictions_path, index=False)

        mlflow.log_artifact(
            str(predictions_path),
            artifact_path="predictions",
        )

    # -------------------------
    # Log the complete pipeline
    # -------------------------

    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="real_estate_price_pipeline",
        signature=signature,
        input_example=input_example,
        registered_model_name=REGISTERED_MODEL_NAME,
        serialization_format="cloudpickle",
    )

    # Log the model URI for easy reference.
    mlflow.log_param("logged_model_uri", model_info.model_uri)

    print(f"MLflow run ID: {run_id}")
    print(f"Logged model URI: {model_info.model_uri}")


        Log-target metrics:
        MAE: 0.1159
        RMSE: 0.1824
        R2: 0.8922

        Original-price metrics:
        MAE: 0.48
        RMSE: 0.99
        R2: 0.8622
        MAPE: 21.63%
        


/mnt/WindowsD/Projects/real-estate-valuation/.estate/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/21 11:48:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, 

MLflow run ID: 0ef617d7b27c4ea58da051576d85b489
Logged model URI: models:/m-4af7cd1970114616bb219815f3646127
🏃 View run xgboost_log_target_pipeline at: https://dagshub.com/jubito27/gurgaon-estate.mlflow/#/experiments/2/runs/0ef617d7b27c4ea58da051576d85b489
🧪 View experiment at: https://dagshub.com/jubito27/gurgaon-estate.mlflow/#/experiments/2
